In [ ]:
# ============================================================
# 02_eda.ipynb
# Exploratory data analysis: sex-stratified diabetes prevalence
# and descriptive statistics supporting the sex-stratified design.
#
# Input : data/processed/df_total.csv, df_male.csv, df_female.csv
# Output: results/tables/prevalence_by_sex.csv,
#         results/tables/prevalence_by_year.csv,
#         results/tables/descriptive_stats.csv,
#         results/figures/eda_prevalence_by_sex.(png|pdf),
#         results/figures/eda_age_distribution.(png|pdf)
#
# Figures: grayscale (seaborn), dpi 600, PNG + PDF, no titles/captions.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ---- Resolve repository root ----
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

# ---- Grayscale publication style (no titles/captions in-image) ----
GRAY = ["#000000", "#595959", "#8C8C8C", "#BFBFBF", "#D9D9D9"]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette(GRAY))
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": "0.88", "grid.linewidth": 0.6,
    "image.cmap": "Greys", "axes.prop_cycle": mpl.cycler(color=GRAY),
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

# ---- Load ----
df = pd.read_csv(PROC / "df_total.csv")
df_m = pd.read_csv(PROC / "df_male.csv")
df_f = pd.read_csv(PROC / "df_female.csv")
num_col = ["HE_BMI", "HE_wc", "HE_wt", "N_SUGAR", "N_CHO", "N_EN", "age"]
TARGET = "HE_DM_HbA1c"

# ---- Table: prevalence by sex ----
rows = []
for name, d in [("Male", df_m), ("Female", df_f), ("Total", df)]:
    n = len(d); pos = int(d[TARGET].sum())
    rows.append({"Sex": name, "N": n,
                 "Non-diabetic n": n - pos, "Non-diabetic %": round((n - pos) / n * 100, 1),
                 "Diabetic n": pos, "Diabetic %": round(pos / n * 100, 1)})
prev = pd.DataFrame(rows)
prev.to_csv(TAB / "prevalence_by_sex.csv", index=False)
print("Prevalence by sex:\n", prev.to_string(index=False))

# ---- Table: prevalence by survey year and sex ----
yr = (df.assign(Sex=df["sex"].map({1: "Male", 2: "Female"}))
        .groupby(["year", "Sex"])[TARGET].agg(["size", "sum"])
        .reset_index().rename(columns={"size": "N", "sum": "Diabetic"}))
yr["Diabetic %"] = (yr["Diabetic"] / yr["N"] * 100).round(1)
yr.to_csv(TAB / "prevalence_by_year.csv", index=False)

# ---- Table: descriptive statistics (continuous) by sex ----
desc_rows = []
for c in num_col:
    desc_rows.append({
        "Variable": c,
        "Male mean (SD)": f"{df_m[c].mean():.1f} ({df_m[c].std():.1f})",
        "Female mean (SD)": f"{df_f[c].mean():.1f} ({df_f[c].std():.1f})",
    })
desc = pd.DataFrame(desc_rows)
desc.to_csv(TAB / "descriptive_stats.csv", index=False)
print("\nDescriptive statistics saved.")

# ---- Figure: diabetes prevalence by sex (grayscale bars) ----
fig, ax = plt.subplots(figsize=(4.2, 3.4))
bars = ax.bar(["Male", "Female"], [df_m[TARGET].mean() * 100, df_f[TARGET].mean() * 100],
              color=[GRAY[1], GRAY[3]], edgecolor="black", width=0.6)
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
            f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Diabetes prevalence (%)")
ax.set_ylim(0, 35)
save_fig(fig, "eda_prevalence_by_sex")

# ---- Figure: age distribution by diabetes status and sex ----
fig, axes = plt.subplots(1, 2, figsize=(8, 3.4), sharey=True)
for ax, d, lab in zip(axes, [df_m, df_f], ["Male", "Female"]):
    for k, (gl, gray) in enumerate([(0, GRAY[3]), (1, GRAY[0])]):
        sns.kdeplot(d.loc[d[TARGET] == gl, "age"], ax=ax, color=gray,
                    fill=(k == 0), alpha=0.5 if k == 0 else 1.0, linewidth=1.6,
                    label=("Non-diabetic" if gl == 0 else "Diabetic"))
    ax.set_xlabel(f"Age (years) — {lab}")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("Density")
save_fig(fig, "eda_age_distribution")

print("EDA figures written to results/figures/ (png + pdf).")
